In [1]:
import pandas as pd
import numpy as np
import json
import re
import time
from pathlib import Path
from collections import Counter
import requests
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

pd.set_option("display.max_colwidth", 120)
pio.templates.default = "plotly_white"

In [2]:
panel = pd.read_csv("interviews.csv")
COMPCOLS = [f"Q_comparaison_{i}" for i in range(1, 7)]
df = panel[["panelist_id"] + COMPCOLS].copy()
print(panel.shape)
df.head(2)

(800, 48)


,panelist_id,Q_comparaison_1,Q_comparaison_2,Q_comparaison_3,Q_comparaison_4,Q_comparaison_5,Q_comparaison_6
0,67b0d135d362f5886c2e5cfe,"J'ai préféré celle de Bouygues. Elle était plus marrante, plus originale. Je l'ai regardée comme un petit sketch, c'...","Celle de Bouygues, sans hésiter. L'idée de la scène de crime pour du wifi, c'est tellement absurde que tu t'en souvi...","Bouygues. Clairement. Pour son concept. Ils ont osé faire un truc complètement différent, une parodie. Ça sort du ca...","Bouygues. Ça m'a fait rire. C'est une émotion simple, mais c'est positif. La pub Orange, elle ne suscite pas vraimen...","C'est marrant parce que même si j'ai préféré la pub Bouygues, je crois que c'est celle d'Orange qui me ferait le plu...","Je dirais Orange. Leur promesse c'est 'la fiabilité', et toute la pub est construite pour te montrer pourquoi c'est ..."
1,67b0d12ed362f5886c2e5b9c,Hmm... difficile. J'ai bien aimé les deux pour des raisons différentes. Mais je vais dire Bouygues. Juste pour l'ori...,"Celle de Bouygues, je pense. L'idée de la 'police du WiFi' c'est un concept fort, facile à retenir et à raconter. La...","Bouygues, sans aucune hésitation. Pour tout le côté parodie de série policière. C'est un vrai parti pris créatif. Or...","Bouygues m'a fait plus rire. Donc si on parle d'émotion forte, le rire, c'est celle-là. Orange, c'est plus une sympa...","Alors là, c'est marrant, mais je dirais peut-être Orange. Même si j'ai préféré la pub Bouygues. Parce que le message...","Je dirais Orange. Leur promesse c'est la fiabilité, et ils le montrent bien avec des exemples où tout le reste échou..."


In [3]:
def build_verbatim(row):
    parts = [str(row[col]) for col in COMPCOLS if pd.notna(row[col])]
    return " ".join(parts)

df["verbatim"] = df.apply(build_verbatim, axis=1)
df[["panelist_id", "verbatim"]].head(3)

,panelist_id,verbatim
0,67b0d135d362f5886c2e5cfe,"J'ai préféré celle de Bouygues. Elle était plus marrante, plus originale. Je l'ai regardée comme un petit sketch, c'..."
1,67b0d12ed362f5886c2e5b9c,Hmm... difficile. J'ai bien aimé les deux pour des raisons différentes. Mais je vais dire Bouygues. Juste pour l'ori...
2,67b0d12bd362f5886c2e5b09,"J'ai trouvé la pub Bouygues très drôle, mais je crois que je préfère celle d'Orange. Elle est plus simple, elle me p..."


In [4]:
SCHEMA_EXTENDED = {
    "creativity": {
        "values": ["Bouygues", "Orange", "Neutral", "Mixed"],
        "rule": "Which brand does the panelist prefer from a pure creativity angle: inventiveness of the concept, boldness of the idea, creative spark? Mixed = both equally creative. Neutral = no clear opinion."
    },
    "humour": {
        "values": ["Bouygues", "Orange", "Neutral", "Mixed"],
        "rule": "Which brand does the panelist find funnier or more entertaining? Mixed = both equally funny. Neutral = neither stands out for humour."
    },
    "originality": {
        "values": ["Bouygues", "Orange", "Neutral", "Mixed"],
        "rule": "Which brand is seen as more original, distinctive, surprising, or memorable in its creative execution? Mixed = both equally original. Neutral = no clear originality signal."
    },
    "reliability_trust": {
        "values": ["Bouygues", "Orange", "Neutral", "Mixed"],
        "rule": "Which brand inspires more reliability, seriousness, reassurance, or trust from the ad? Mixed = both equally trustworthy. Neutral = no clear signal."
    },
    "intent_to_purchase": {
        "values": ["Bouygues", "Orange", "Neutral", "Mixed"],
        "rule": "Which brand makes the panelist more likely to subscribe, switch, or purchase? Mixed = equal purchase intent. Neutral = neither drives intent."
    },
    "overallpreference": {
        "values": ["Bouygues", "Orange", "Neutral", "Mixed"],
        "rule": "Which brand is preferred overall, taking everything into account? Mixed = genuinely undecided overall. Neutral = no overall preference expressed."
    }
}

DIM_GROUPS = {
    "creative": ["creativity", "humour", "originality"],
    "commercial": ["reliability_trust", "intent_to_purchase"],
    "overall": ["overallpreference"]
}

ALL_DIMS = [d for dims in DIM_GROUPS.values() for d in dims]
ALL_DIMS

['creativity',
 'humour',
 'originality',
 'reliability_trust',
 'intent_to_purchase',
 'overallpreference']

In [5]:
SYSTEM_PROMPT_EXT = """Tu es un annotateur expert en analyse de discours publicitaire.
Tu analyses des verbatims de panélistes ayant regardé deux publicités télévisées :
- Bouygues Telecom : humour absurde, scénario policier décalé
- Orange : message de fiabilité réseau, ton rassurant

Tu dois extraire 6 dimensions de préférence à partir du verbatim fourni.

SCHÉMA D'ANNOTATION
{schema}

RÈGLES STRICTES
1. Réponds UNIQUEMENT avec un objet JSON valide, sans texte avant ou après.
2. Chaque valeur doit être exactement l'une de : Bouygues, Orange, Neutral, Mixed.
3. Ajoute un champ "reasoning" très court expliquant les choix.
4. Ajoute un champ "confidence" entre 0.0 et 1.0.

FORMAT DE SORTIE
{{
{fields}
  "reasoning": "explication courte",
  "confidence": 0.95
}}
""".format(
    schema="\n".join(f"- {dim}: {info['rule']}" for dim, info in SCHEMA_EXTENDED.items()),
    fields="\n".join(f'  "{dim}": "Bouygues",' for dim in SCHEMA_EXTENDED)
)

FEW_SHOT_EXT = [
    {
        "verbatim": "J'ai préféré celle de Bouygues. Elle était plus drôle, plus originale. Je l'ai trouvée plus inventive. Mais pour choisir un opérateur, Orange me rassure davantage sur la fiabilité et me donnerait plus envie de souscrire.",
        "label": {
            "creativity": "Bouygues",
            "humour": "Bouygues",
            "originality": "Bouygues",
            "reliability_trust": "Orange",
            "intent_to_purchase": "Orange",
            "overallpreference": "Mixed"
        }
    },
    {
        "verbatim": "Bouygues, sans hésiter. Le concept est plus fort, plus drôle, plus original. Et au final la marque me donne aussi confiance, donc c'est celle que je choisirais.",
        "label": {
            "creativity": "Bouygues",
            "humour": "Bouygues",
            "originality": "Bouygues",
            "reliability_trust": "Bouygues",
            "intent_to_purchase": "Bouygues",
            "overallpreference": "Bouygues"
        }
    },
    {
        "verbatim": "Je préfère Orange. C'est moins drôle, mais plus concret, plus crédible, plus rassurant. C'est clairement celle qui me donnerait envie de changer d'opérateur.",
        "label": {
            "creativity": "Neutral",
            "humour": "Neutral",
            "originality": "Neutral",
            "reliability_trust": "Orange",
            "intent_to_purchase": "Orange",
            "overallpreference": "Orange"
        }
    }
]

In [6]:
def extract_json(text):
    try:
        m = re.search(r"\{.*\}", text, re.DOTALL)
        return json.loads(m.group()) if m else None
    except Exception:
        return None

LABEL_VALS = ["Bouygues", "Orange", "Neutral", "Mixed"]
CONFIDENCE_THRESHOLD = 0.7

def build_user_prompt_ext(verbatim):
    prompt = "EXEMPLES ANNOTÉS\n"
    for ex in FEW_SHOT_EXT:
        prompt += f"Verbatim: {ex['verbatim']}\n"
        prompt += f"Annotation: {json.dumps(ex['label'], ensure_ascii=False)}\n---\n"
    prompt += f"\nVERBATIM À ANNOTER\n{verbatim}"
    return prompt

def validate_labels_ext(parsed):
    for dim in SCHEMA_EXTENDED:
        if parsed.get(dim) not in LABEL_VALS:
            parsed[dim] = "ParseError"
    return parsed

def annotate_verbatim_ext(verbatim):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT_EXT},
        {"role": "user", "content": build_user_prompt_ext(verbatim)}
    ]
    resp = requests.post(
        "http://localhost:11434/api/chat",
        json={
            "model": "mistral",
            "messages": messages,
            "stream": False,
            "options": {"temperature": 0.1}
        },
        timeout=120
    )
    generated = resp.json()["message"]["content"]
    parsed = extract_json(generated)
    if parsed is None:
        out = {dim: "ParseError" for dim in SCHEMA_EXTENDED}
        out["reasoning"] = generated[:200]
        out["confidence"] = 0.0
        out["raw_output"] = generated
        return out
    parsed = validate_labels_ext(parsed)
    confidence = float(parsed.get("confidence", 0.0))
    for dim in SCHEMA_EXTENDED:
        parsed[f"{dim}_flagged"] = confidence < CONFIDENCE_THRESHOLD
    parsed["raw_output"] = generated
    return parsed

In [7]:
test = annotate_verbatim_ext("J'ai préféré Bouygues pour l'humour et l'originalité, mais Orange me rassure davantage et me donnerait plus envie de m'abonner.")
{k: v for k, v in test.items() if k != "raw_output"}

{'creativity': 'Bouygues',
 'humour': 'Bouygues',
 'originality': 'Bouygues',
 'reliability_trust': 'Orange',
 'intent_to_purchase': 'Orange',
 'overallpreference': 'Mixed',
 'reasoning': 'The panelist prefers Bouygues for its humor and originality, but Orange is seen as more trustworthy and likely to encourage subscription.',
 'confidence': 0.95,
 'creativity_flagged': False,
 'humour_flagged': False,
 'originality_flagged': False,
 'reliability_trust_flagged': False,
 'intent_to_purchase_flagged': False,
 'overallpreference_flagged': False}

In [8]:
# PILOTN = 30
# pilot_df = df.sample(n=PILOTN, random_state=42).copy().reset_index(drop=True)

# pilot_results = []
# for i, row in pilot_df.iterrows():
#     t0 = time.time()
#     result = annotate_verbatim_ext(row["verbatim"])
#     result["panelist_id"] = row["panelist_id"]
#     pilot_results.append(result)
#     elapsed = time.time() - t0
#     print(f"{i+1:02d}/{PILOTN} {elapsed:.1f}s "
#           f"creativity={result['creativity']} humour={result['humour']} originality={result['originality']} "
#           f"trust={result['reliability_trust']} purchase={result['intent_to_purchase']} overall={result['overallpreference']} "
#           f"conf={result.get('confidence', '?')}")
# pilot_results = pd.DataFrame(pilot_results)
# pilot_results.to_csv("pilot_annotations_extended.csv", index=False)
# pilot_results.head()

In [9]:
# CHECKPOINT_EVERY = 50
# checkpoint_path = "annotations_extended_checkpoint.csv"
# all_results = []
# run_start = time.time()
# total = len(df)

# for i, row in df.iterrows():
#     t0 = time.time()
#     result = annotate_verbatim_ext(row["verbatim"])
#     result["panelist_id"] = row["panelist_id"]
#     all_results.append(result)

#     if (i + 1) % CHECKPOINT_EVERY == 0:
#         pd.DataFrame(all_results).to_csv(checkpoint_path, index=False)
#         done = i + 1
#         avg = (time.time() - run_start) / done
#         eta = avg * (total - done) / 60
#         print(f"{done}/{total} checkpoint saved avg {avg:.1f}s/sample ETA {eta:.0f} min")

# annotations_ext = pd.DataFrame(all_results)
# annotations_ext.to_csv("preference_annotations_extended.csv", index=False)
# annotations_ext.head()

In [10]:
final = pd.read_csv("preference_annotations_extended.csv")
total = len(final)

for dim in SCHEMA_EXTENDED:
    errors = (final[dim] == "ParseError").sum()
    flagged = final[f"{dim}_flagged"].sum() if f"{dim}_flagged" in final.columns else np.nan
    dist = final[dim].value_counts(normalize=True).round(3).to_dict()
    print(dim)
    print("ParseErrors:", errors, f"({errors/total:.1%})")
    print("Flagged:", flagged)
    print("Distribution:", dist)
    print()

print("Mean confidence:", round(final["confidence"].mean(), 3))

creativity
ParseErrors: 0 (0.0%)
Flagged: 0
Distribution: {'Bouygues': 0.904, 'Orange': 0.055, 'Mixed': 0.041}

humour
ParseErrors: 0 (0.0%)
Flagged: 0
Distribution: {'Bouygues': 0.445, 'Orange': 0.29, 'Neutral': 0.229, 'Mixed': 0.036}

originality
ParseErrors: 0 (0.0%)
Flagged: 0
Distribution: {'Bouygues': 0.864, 'Mixed': 0.128, 'Orange': 0.009}

reliability_trust
ParseErrors: 0 (0.0%)
Flagged: 0
Distribution: {'Orange': 0.871, 'Neutral': 0.085, 'Bouygues': 0.036, 'Mixed': 0.008}

intent_to_purchase
ParseErrors: 0 (0.0%)
Flagged: 0
Distribution: {'Orange': 0.78, 'Bouygues': 0.152, 'Mixed': 0.068}

overallpreference
ParseErrors: 0 (0.0%)
Flagged: 0
Distribution: {'Orange': 0.506, 'Mixed': 0.289, 'Bouygues': 0.202, 'Neutral': 0.002}

Mean confidence: 0.94


In [11]:
cols_to_merge = ["panelist_id"] + ALL_DIMS + ["confidence", "reasoning"]
panel_merged = panel.merge(final[cols_to_merge], on="panelist_id", how="left")
panel_merged.to_csv("interviews_with_extended_preferences.csv", index=False)
print(panel_merged.shape)
panel_merged[["panelist_id"] + ALL_DIMS + ["confidence"]].head()

(800, 56)


,panelist_id,creativity,humour,originality,reliability_trust,intent_to_purchase,overallpreference,confidence
0,67b0d135d362f5886c2e5cfe,Bouygues,Bouygues,Bouygues,Orange,Mixed,Bouygues,0.95
1,67b0d12ed362f5886c2e5b9c,Bouygues,Bouygues,Bouygues,Orange,Orange,Mixed,0.95
2,67b0d12bd362f5886c2e5b09,Bouygues,Bouygues,Bouygues,Orange,Orange,Mixed,0.95
3,67b0d127d362f5886c2e5a4a,Bouygues,Neutral,Bouygues,Orange,Orange,Orange,0.95
4,67b0d12cd362f5886c2e5b4a,Bouygues,Bouygues,Bouygues,Orange,Orange,Mixed,0.95


In [12]:
summary = pd.DataFrame({
    "Dimension": ALL_DIMS,
    "Bouygues": [100 * (final[d] == "Bouygues").mean() for d in ALL_DIMS],
    "Orange": [100 * (final[d] == "Orange").mean() for d in ALL_DIMS],
    "Mixed": [100 * (final[d] == "Mixed").mean() for d in ALL_DIMS],
    "Neutral": [100 * (final[d] == "Neutral").mean() for d in ALL_DIMS]
}).set_index("Dimension").round(1)

summary.to_csv("preference_summary_extended.csv")
summary

,Bouygues,Orange,Mixed,Neutral
Dimension,,,,
creativity,90.4,5.5,4.1,0.0
humour,44.5,29.0,3.6,22.9
originality,86.4,0.9,12.8,0.0
reliability_trust,3.6,87.1,0.8,8.5
intent_to_purchase,15.2,78.0,6.8,0.0
overallpreference,20.2,50.6,28.9,0.2


In [13]:
colors = {
    "Bouygues": "#0055A4",
    "Orange": "#FF6600",
    "Mixed": "#888888",
    "Neutral": "#CCCCCC"
}

dim_labels = {
    "creativity": "Creativity",
    "humour": "Humour",
    "originality": "Originality",
    "reliability_trust": "Reliability & trust",
    "intent_to_purchase": "Intent to purchase",
    "overallpreference": "Overall"
}

legend_below = dict(orientation="h", yanchor="top", y=-0.18, xanchor="center", x=0.5)
margins = dict(t=80, b=120, l=60, r=20)

In [14]:
%pip install -U nbformat

Note: you may need to restart the kernel to use updated packages.


In [15]:
labels_order = ["Bouygues", "Orange", "Mixed", "Neutral"]
fig = go.Figure()

for label in labels_order:
    vals = [(final[d] == label).mean() * 100 for d in ALL_DIMS]
    fig.add_trace(go.Bar(
        name=label,
        x=[dim_labels[d] for d in ALL_DIMS],
        y=vals,
        marker_color=colors[label]
    ))

fig.update_layout(
    barmode="stack",
    title=dict(text="Preference distribution across all dimensions", x=0, xanchor="left"),
    legend=legend_below,
    margin=margins
)
fig.update_yaxes(title_text="% of panelists")
fig.update_xaxes(title_text="")
fig.show()

In [21]:
DEMOGRAPHICS = {
    "gender": "Gender",
    "age_group": "Age",
    "csp": "CSP",
    "income_level": "Income level",
    "education": "Education",
    "location.city_size": "City size"
}

available_demographics = {k: v for k, v in DEMOGRAPHICS.items() if k in panel_merged.columns}
available_demographics

{'gender': 'Gender',
 'age_group': 'Age',
 'csp': 'CSP',
 'income_level': 'Income level',
 'education': 'Education',
 'location.city_size': 'City size'}

In [22]:
panel_merged["age_group"] = pd.cut(
    panel_merged["age"],
    bins=[17, 24, 34, 44, 54, 64, 120],
    labels=["18-24", "25-34", "35-44", "45-54", "55-64", "65+"]
) if "age" in panel_merged.columns else np.nan

In [23]:
def prep_demo_col(df, col, min_count=10, max_levels=12):
    out = df.copy()
    out = out[out[col].notna()].copy()
    out["_demo"] = out[col].astype(str)
    out = out[out["_demo"] != "nan"].copy()

    counts = out["_demo"].value_counts()
    keep = counts[counts >= min_count].index.tolist()

    if len(keep) == 0:
        keep = counts.index.tolist()

    out["_demo"] = np.where(out["_demo"].isin(keep), out["_demo"], "Other")

    counts2 = out["_demo"].value_counts()
    top_levels = counts2.index.tolist()[:max_levels]
    out = out[out["_demo"].isin(top_levels)].copy()

    return out

In [24]:
def make_facet_plot(df, preference_col, demographic_col, demographic_label):
    plot_df = prep_demo_col(df, demographic_col)

    ct = (
        plot_df.groupby("_demo")[preference_col]
        .value_counts(normalize=True)
        .mul(100)
        .rename("pct")
        .reset_index()
    )

    ct = ct.rename(columns={"_demo": demographic_label, preference_col: "preference"})
    ct = ct[ct["preference"].isin(["Bouygues", "Orange", "Mixed", "Neutral"])].copy()

    n_levels = ct[demographic_label].nunique()
    n_cols = min(3, max(1, n_levels))
    n_rows = int(np.ceil(n_levels / n_cols))

    fig = px.bar(
        ct,
        x="preference",
        y="pct",
        color="preference",
        facet_col=demographic_label,
        facet_col_wrap=n_cols,
        category_orders={"preference": ["Bouygues", "Orange", "Mixed", "Neutral"]},
        color_discrete_map=colors,
        title=f"{dim_labels[preference_col]} by {demographic_label}"
    )

    fig.update_layout(
        showlegend=False,
        margin=dict(t=80, b=50, l=50, r=20),
        height=max(450, 260 * n_rows)
    )

    fig.update_yaxes(title_text="% of panelists", matches=None)
    fig.update_xaxes(title_text="")

    return fig

In [25]:
for pref in ALL_DIMS:
    for demo_col, demo_label in available_demographics.items():
        fig = make_facet_plot(panel_merged, pref, demo_col, demo_label)
        fig.show()

In [26]:
output_dir = Path("extended_pref_charts")
output_dir.mkdir(exist_ok=True)

In [27]:
saved_files = []

for pref in ALL_DIMS:
    for demo_col, demo_label in available_demographics.items():
        fig = make_facet_plot(panel_merged, pref, demo_col, demo_label)
        fname = f"{pref}__by__{demo_col.replace('.', '_')}.png"
        path = output_dir / fname
        fig.write_image(str(path), width=1600, height=900, scale=2)
        saved_files.append(str(path))

len(saved_files)

36

In [28]:
def make_long_table(df):
    rows = []
    for pref in ALL_DIMS:
        for demo_col, demo_label in available_demographics.items():
            temp = prep_demo_col(df, demo_col)
            temp_demo_name = "age_group" if demo_col == "age" else demo_col
            ct = (
                temp.groupby("_demo")[pref]
                .value_counts(normalize=True)
                .mul(100)
                .rename("pct")
                .reset_index()
            )
            ct["dimension"] = pref
            ct["dimension_label"] = dim_labels[pref]
            ct["demographic"] = temp_demo_name
            ct["demographic_label"] = demo_label
            ct = ct.rename(columns={"_demo": "segment", pref: "preference"})
            rows.append(ct)
    return pd.concat(rows, ignore_index=True)

long_table = make_long_table(panel_merged)
long_table.to_csv("extended_facet_table.csv", index=False)
long_table.head()

,segment,preference,pct,dimension,dimension_label,demographic,demographic_label
0,Female,Bouygues,90.322581,creativity,Creativity,gender,Gender
1,Female,Orange,5.806452,creativity,Creativity,gender,Gender
2,Female,Mixed,3.870968,creativity,Creativity,gender,Gender
3,Male,Bouygues,89.277389,creativity,Creativity,gender,Gender
4,Male,Orange,5.827506,creativity,Creativity,gender,Gender


In [29]:
for pref in ["creativity", "humour", "originality", "reliability_trust", "intent_to_purchase", "overallpreference"]:
    display(long_table[long_table["dimension"] == pref].head(20))

,segment,preference,pct,dimension,dimension_label,demographic,demographic_label
0,Female,Bouygues,90.322581,creativity,Creativity,gender,Gender
1,Female,Orange,5.806452,creativity,Creativity,gender,Gender
2,Female,Mixed,3.870968,creativity,Creativity,gender,Gender
3,Male,Bouygues,89.277389,creativity,Creativity,gender,Gender
4,Male,Orange,5.827506,creativity,Creativity,gender,Gender
5,Male,Mixed,4.895105,creativity,Creativity,gender,Gender
6,Non-binary,Bouygues,98.360656,creativity,Creativity,gender,Gender
7,Non-binary,Orange,1.639344,creativity,Creativity,gender,Gender
8,18-24,Bouygues,98.165138,creativity,Creativity,age_group,Age
9,18-24,Mixed,0.917431,creativity,Creativity,age_group,Age


,segment,preference,pct,dimension,dimension_label,demographic,demographic_label
91,Female,Bouygues,51.290323,humour,Humour,gender,Gender
92,Female,Orange,29.677419,humour,Humour,gender,Gender
93,Female,Neutral,16.129032,humour,Humour,gender,Gender
94,Female,Mixed,2.903226,humour,Humour,gender,Gender
95,Male,Bouygues,34.731935,humour,Humour,gender,Gender
96,Male,Orange,31.468531,humour,Humour,gender,Gender
97,Male,Neutral,29.370629,humour,Humour,gender,Gender
98,Male,Mixed,4.428904,humour,Humour,gender,Gender
99,Non-binary,Bouygues,78.688525,humour,Humour,gender,Gender
100,Non-binary,Neutral,11.475410,humour,Humour,gender,Gender


,segment,preference,pct,dimension,dimension_label,demographic,demographic_label
227,Female,Bouygues,89.354839,originality,Originality,gender,Gender
228,Female,Mixed,10.645161,originality,Originality,gender,Gender
229,Male,Bouygues,82.983683,originality,Originality,gender,Gender
230,Male,Mixed,15.617716,originality,Originality,gender,Gender
231,Male,Orange,1.398601,originality,Originality,gender,Gender
232,Non-binary,Bouygues,95.081967,originality,Originality,gender,Gender
233,Non-binary,Mixed,3.278689,originality,Originality,gender,Gender
234,Non-binary,Orange,1.639344,originality,Originality,gender,Gender
235,18-24,Bouygues,96.330275,originality,Originality,age_group,Age
236,18-24,Mixed,2.752294,originality,Originality,age_group,Age


,segment,preference,pct,dimension,dimension_label,demographic,demographic_label
314,Female,Orange,87.419355,reliability_trust,Reliability & trust,gender,Gender
315,Female,Neutral,8.387097,reliability_trust,Reliability & trust,gender,Gender
316,Female,Bouygues,2.580645,reliability_trust,Reliability & trust,gender,Gender
317,Female,Mixed,1.612903,reliability_trust,Reliability & trust,gender,Gender
318,Male,Orange,89.510490,reliability_trust,Reliability & trust,gender,Gender
319,Male,Neutral,6.759907,reliability_trust,Reliability & trust,gender,Gender
320,Male,Bouygues,3.496503,reliability_trust,Reliability & trust,gender,Gender
321,Male,Mixed,0.233100,reliability_trust,Reliability & trust,gender,Gender
322,Non-binary,Orange,68.852459,reliability_trust,Reliability & trust,gender,Gender
323,Non-binary,Neutral,21.311475,reliability_trust,Reliability & trust,gender,Gender


,segment,preference,pct,dimension,dimension_label,demographic,demographic_label
426,Female,Orange,74.516129,intent_to_purchase,Intent to purchase,gender,Gender
427,Female,Bouygues,16.451613,intent_to_purchase,Intent to purchase,gender,Gender
428,Female,Mixed,9.032258,intent_to_purchase,Intent to purchase,gender,Gender
429,Male,Orange,83.916084,intent_to_purchase,Intent to purchase,gender,Gender
430,Male,Bouygues,11.888112,intent_to_purchase,Intent to purchase,gender,Gender
431,Male,Mixed,4.195804,intent_to_purchase,Intent to purchase,gender,Gender
432,Non-binary,Orange,54.098361,intent_to_purchase,Intent to purchase,gender,Gender
433,Non-binary,Bouygues,32.786885,intent_to_purchase,Intent to purchase,gender,Gender
434,Non-binary,Mixed,13.114754,intent_to_purchase,Intent to purchase,gender,Gender
435,18-24,Orange,46.788991,intent_to_purchase,Intent to purchase,age_group,Age


,segment,preference,pct,dimension,dimension_label,demographic,demographic_label
523,Female,Orange,43.870968,overallpreference,Overall,gender,Gender
524,Female,Mixed,33.548387,overallpreference,Overall,gender,Gender
525,Female,Bouygues,22.258065,overallpreference,Overall,gender,Gender
526,Female,Neutral,0.322581,overallpreference,Overall,gender,Gender
527,Male,Orange,61.305361,overallpreference,Overall,gender,Gender
528,Male,Mixed,23.310023,overallpreference,Overall,gender,Gender
529,Male,Bouygues,15.384615,overallpreference,Overall,gender,Gender
530,Non-binary,Bouygues,44.262295,overallpreference,Overall,gender,Gender
531,Non-binary,Mixed,44.262295,overallpreference,Overall,gender,Gender
532,Non-binary,Orange,9.836066,overallpreference,Overall,gender,Gender


In [30]:
def make_grouped_dimension_plot(df, group_name, demographic_col, demographic_label):
    dims = DIM_GROUPS[group_name]
    temp = prep_demo_col(df, demographic_col)
    pieces = []
    for d in dims:
        ct = (
            temp.groupby("_demo")[d]
            .value_counts(normalize=True)
            .mul(100)
            .rename("pct")
            .reset_index()
        )
        ct = ct.rename(columns={"_demo": demographic_label, d: "preference"})
        ct["dimension"] = dim_labels[d]
        pieces.append(ct)
    plot_df = pd.concat(pieces, ignore_index=True)
    plot_df = plot_df[plot_df["preference"].isin(["Bouygues", "Orange", "Mixed", "Neutral"])]
    fig = px.bar(
        plot_df,
        x="preference",
        y="pct",
        color="preference",
        facet_row="dimension",
        facet_col=demographic_label,
        facet_col_wrap=3,
        category_orders={"preference": ["Bouygues", "Orange", "Mixed", "Neutral"]},
        color_discrete_map=colors,
        title=f"{group_name.capitalize()} preferences by {demographic_label}"
    )
    fig.update_layout(margin=dict(t=100, b=60, l=50, r=20))
    fig.update_yaxes(title_text="% of panelists")
    fig.update_xaxes(title_text="")
    return fig

In [31]:
for group_name in DIM_GROUPS:
    for demo_col, demo_label in available_demographics.items():
        fig = make_grouped_dimension_plot(panel_merged, group_name, demo_col, demo_label)
        fig.show()

In [32]:
grouped_output_dir = Path("grouped_facet_charts")
grouped_output_dir.mkdir(exist_ok=True)

grouped_saved_files = []

for group_name in DIM_GROUPS:
    for demo_col, demo_label in available_demographics.items():
        fig = make_grouped_dimension_plot(panel_merged, group_name, demo_col, demo_label)
        fname = f"{group_name}__facets__{demo_col.replace('.', '_')}.png"
        path = grouped_output_dir / fname
        fig.write_image(str(path), width=1800, height=1200, scale=2)
        grouped_saved_files.append(str(path))

len(grouped_saved_files)

18

In [33]:
panel_merged.to_csv("interviews_with_extended_preferences.csv", index=False)
summary.to_csv("preference_summary_extended.csv")
print("done")

done


In [34]:
from pathlib import Path
import math
import plotly.express as px
import plotly.graph_objects as go

base_dir = Path("charts")
by_demo_dir = base_dir / "by_demographic"
hist_dir = base_dir / "histograms"
donut_dir = base_dir / "donuts"

for d in [base_dir, by_demo_dir, hist_dir, donut_dir]:
    d.mkdir(parents=True, exist_ok=True)

def slugify(s):
    return (
        str(s)
        .lower()
        .replace(" & ", "_")
        .replace(".", "_")
        .replace("/", "_")
        .replace(" ", "_")
    )

def save_fig(fig, path, width=2200, height=1200, scale=2):
    fig.write_image(str(path), width=width, height=height, scale=scale)

def make_histogram_plot(df, preference_col):
    counts = (
        df[preference_col]
        .value_counts(dropna=False)
        .reindex(["Bouygues", "Orange", "Mixed", "Neutral"], fill_value=0)
        .reset_index()
    )
    counts.columns = ["preference", "n"]

    fig = px.bar(
        counts,
        x="preference",
        y="n",
        color="preference",
        category_orders={"preference": ["Bouygues", "Orange", "Mixed", "Neutral"]},
        color_discrete_map=colors,
        title=f"{dim_labels[preference_col]} distribution"
    )
    fig.update_layout(
        showlegend=False,
        width=1600,
        height=900,
        margin=dict(t=90, b=70, l=70, r=30)
    )
    fig.update_yaxes(title_text="Count")
    fig.update_xaxes(title_text="")
    return fig

def make_donut_plot(df, preference_col):
    counts = (
        df[preference_col]
        .value_counts(dropna=False)
        .reindex(["Bouygues", "Orange", "Mixed", "Neutral"], fill_value=0)
    )

    fig = go.Figure(
        go.Pie(
            labels=counts.index.tolist(),
            values=counts.values.tolist(),
            hole=0.5,
            marker_colors=[colors[k] for k in counts.index.tolist()],
            sort=False
        )
    )
    fig.update_traces(textinfo="percent+label")
    fig.update_layout(
        title=f"{dim_labels[preference_col]} share",
        width=1400,
        height=900,
        margin=dict(t=90, b=60, l=40, r=40),
        legend=dict(orientation="h", yanchor="top", y=-0.05, xanchor="center", x=0.5)
    )
    return fig

saved = []

for demo_col, demo_label in available_demographics.items():
    demo_folder = by_demo_dir / slugify(demo_col)
    demo_folder.mkdir(parents=True, exist_ok=True)

    for pref in ALL_DIMS:
        fig = make_facet_plot(panel_merged, pref, demo_col, demo_label)

        plot_df = prep_demo_col(panel_merged, demo_col)
        n_levels = plot_df["_demo"].nunique() if len(plot_df) else 1
        n_cols = min(3, max(1, n_levels))
        n_rows = max(1, math.ceil(n_levels / n_cols))

        fig.update_layout(
            width=2400,
            height=max(1000, 420 * n_rows),
            margin=dict(t=100, b=80, l=70, r=40),
            title_x=0.01,
            font=dict(size=16)
        )
        fig.update_annotations(font_size=15)
        fig.update_yaxes(title_text="% of panelists")
        fig.update_xaxes(title_text="")

        out = demo_folder / f"{slugify(pref)}__by__{slugify(demo_col)}.png"
        save_fig(fig, out, width=2400, height=max(1000, 420 * n_rows), scale=2)
        saved.append(str(out))

for pref in ALL_DIMS:
    fig_h = make_histogram_plot(panel_merged, pref)
    out_h = hist_dir / f"histogram__{slugify(pref)}.png"
    save_fig(fig_h, out_h, width=1600, height=900, scale=2)
    saved.append(str(out_h))

    fig_d = make_donut_plot(panel_merged, pref)
    out_d = donut_dir / f"donut__{slugify(pref)}.png"
    save_fig(fig_d, out_d, width=1400, height=900, scale=2)
    saved.append(str(out_d))

print(f"Saved {len(saved)} charts")
for s in saved[:10]:
    print(s)
print("...")
print(f"Root folder: {base_dir.resolve()}")

Saved 48 charts
charts/by_demographic/gender/creativity__by__gender.png
charts/by_demographic/gender/humour__by__gender.png
charts/by_demographic/gender/originality__by__gender.png
charts/by_demographic/gender/reliability_trust__by__gender.png
charts/by_demographic/gender/intent_to_purchase__by__gender.png
charts/by_demographic/gender/overallpreference__by__gender.png
charts/by_demographic/age_group/creativity__by__age_group.png
charts/by_demographic/age_group/humour__by__age_group.png
charts/by_demographic/age_group/originality__by__age_group.png
charts/by_demographic/age_group/reliability_trust__by__age_group.png
...
Root folder: /Users/martateodoratrales/Desktop/AirPanel/charts
